# Inference: RF-DETR ONNX + SAHI — confidence threshold sweep
# Version 4. Sarah RFDETRNano model trained 1.7.1
RF-DETR Object Detection on 42 MP drone images.  
Model format: ONNX (exported from .pt checkpoint).  

**UPDATE** Slicing: SAHI `get_sliced_prediction`, 384×384 tiles, 20% overlap.

**update** This version has all diagnostics, convert to coco, removed.

Inference runs **once** at a low base threshold.  
mAP and confusion matrix are computed for each value in `SWEEP_THRESHOLDS` without re-running inference.

**Run order:** Cells 1–3 → **Restart session** → Cells 4 onward.

In [ ]:
!nvidia-smi

In [ ]:
# All pip installs in one place.
# After this cell completes, go to Runtime → Restart session
# (do NOT click 'Restart and run all').
# After the restart, continue running from Cell 4 onward.
%pip install -q rfdetr supervision roboflow
%pip install -q 'sahi[roboflow]'
%pip install -q 'rfdetr[onnxexport]'
%pip uninstall -y onnxruntime onnxruntime-gpu -q
%pip install -q 'onnxruntime-gpu==1.19.2'
%pip install -q 'Pillow==10.4.0' # fix a Pillow version issue
print('Done. Now restart the runtime: Runtime → Restart session')

## ⚠️ Restart the session now

Go to **Runtime → Restart session** (do NOT click 'Restart and run all').  
After the restart, continue running from **Cell 5** downward.

In [ ]:
# Run this AFTER restarting the session to confirm the GPU provider is available. Cell 5
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

import onnxruntime as ort
print('ORT version:', ort.__version__)
print('Available providers:', ort.get_available_providers())

assert 'CUDAExecutionProvider' in ort.get_available_providers(), \
    'CUDAExecutionProvider not available — check onnxruntime-gpu install'

print('PyTorch:', torch.__version__)
%pip show rfdetr

In [ ]:
## Global config — edit paths and sweep values here

# Base threshold for inference — must be <= the lowest value in SWEEP_THRESHOLDS.
# Detections below this score are discarded at SAHI inference time.
# Set to 0.00 to capture everything (SAHI still needs a small positive floor;
# 0.001 is used internally so 0.00 in the sweep is evaluated by keeping all raw preds).
SWEEP_BASE_THRESHOLD = 0.10

# Confidence thresholds to evaluate.
# Inference runs once at SWEEP_BASE_THRESHOLD; these are applied at analysis time.
SWEEP_THRESHOLDS = [0.10, 0.20, 0.25, 0.30, 0.35, 0.40,
                    0.45, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

# IoU threshold for TP/FP/FN matching in the confusion matrix.
# Independent of confidence — 0.5 is the standard COCO value.
IOU_THRESHOLD = 0.5

SLICE_SIZE = 384   # 896 matches model training resolution

IMAGE_SIZE = 384   # Image size used when model was trained. Usually = SLICE_SIZE

OVERLAP_RATIO = 0.20 # SAHI overlap ratio, same used for height, width

#PT_WEIGHTS        = '/content/drive/MyDrive/uavs2/checkpoint_best_total.pth'
PT_WEIGHTS        = '/content/drive/MyDrive/uavs2/sarah_checkpoint_best_total.pth'
ONNX_MODEL_PATH   = '/content/drive/MyDrive/uavs2/sara_rf_detr_nano.onnx'
ANNOTATIONS_PATH  = '/content/datasets/test/_annotations.coco.json'
TEST_IMAGES_DIR   = '/content/datasets/test/images'
OUTPUT_DIR        = '/content/datasets/output/inference'
DRIVE_OUTPUT      = '/content/drive/MyDrive/uavs2/nano_inference_results_onnx_sweep'

CSV_PATH          = f'{OUTPUT_DIR}/threshold_sweep_RF-DETR_nano_detector_extended.csv'

In [ ]:
## CELL 6 — Mount Drive and copy data to Colab
from google.colab import drive
import os, subprocess, shutil

home = os.getcwd()
print(home)

drive.mount('/content/drive')

os.makedirs('/content/datasets/test/images', exist_ok=True)
os.makedirs(OUTPUT_DIR,   exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy images from Drive → local
subprocess.run([
    'rsync', '-av', '--progress',
    '/content/drive/MyDrive/uavs2/images/',
    '/content/datasets/test/images/'
], check=True)

# Copy annotations from Drive → local
shutil.copy(
    '/content/drive/MyDrive/uavs2/_annotations.coco.json',
    ANNOTATIONS_PATH
)

# Copy configuration yaml file for yolo -- also contains classification list
shutil.copy(
    '/content/drive/MyDrive/uavs2/data.yaml',
    '/content/datasets/data.yaml'
)

ann_exists = os.path.exists(ANNOTATIONS_PATH)
img_count  = len(os.listdir(TEST_IMAGES_DIR))
print(f'Annotations found: {ann_exists}')
print(f'Images copied:     {img_count}')

In [ ]:
# utility to clean gpu memory, leaving as much as possible for processing
import gc, torch, weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):
    if not torch.cuda.is_available():
        if verbose:
            print('[INFO] CUDA not available. No GPU cleanup needed.')
        return
    def get_memory_stats():
        return torch.cuda.memory_allocated(), torch.cuda.memory_reserved()
    torch.cuda.synchronize()
    if verbose:
        alloc, reserv = get_memory_stats()
        print(f'[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB')
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print('[WARNING] Object not fully garbage collected yet.')
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    if verbose:
        alloc, reserv = get_memory_stats()
        print(f'[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB')

In [ ]:
## Export .pt checkpoint → ONNX.
## Skipped automatically if the .onnx already exists on Drive.
import os, json, shutil
from rfdetr import RFDETRNano

with open(ANNOTATIONS_PATH) as f:
    coco_meta = json.load(f)

CLASS_NAMES      = [cat['name'] for cat in sorted(coco_meta['categories'], key=lambda c: c['id'])]
CATEGORY_MAPPING = {str(i): name for i, name in enumerate(CLASS_NAMES)}
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')
print(f'CATEGORY_MAPPING: {CATEGORY_MAPPING}')

if os.path.exists(ONNX_MODEL_PATH):
    print(f'\nONNX already exists, skipping export:\n  {ONNX_MODEL_PATH}')
else:
    print(f'\nExporting {PT_WEIGHTS} → ONNX ...')
    export_model = RFDETRNano(
        pretrain_weights=PT_WEIGHTS,
        resolution=IMAGE_SIZE,
        num_classes=len(CLASS_NAMES),
    )
    export_model.export()
    shutil.copy('output/rfdetr-nano.onnx', ONNX_MODEL_PATH)
    del export_model
    cleanup_gpu_memory(verbose=True)
    print(f'Exported and saved to {ONNX_MODEL_PATH}')

In [ ]:
## Load the ONNX model into a custom SAHI DetectionModel subclass.
## detection_model and sahi_result_to_sv_detections() are used by all cells below.

import torch, numpy as np, onnxruntime as ort, supervision as sv
from PIL import Image
from sahi.models.base import DetectionModel
from sahi.prediction import ObjectPrediction
from sahi.predict import get_sliced_prediction


class RFDetrOnnxDetectionModel(DetectionModel):
    """SAHI wrapper for RF-DETR ONNX.
    ONNX outputs (rfdetr 1.5.1):
      dets:   [1, 300, 4]  normalised cx,cy,w,h
      labels: [1, 300, 1]  raw logits  (sigmoid → confidence)
    """

    def load_model(self):
        providers = (
            ['CUDAExecutionProvider', 'CPUExecutionProvider']
            if torch.cuda.is_available() else ['CPUExecutionProvider']
        )
        self.session      = ort.InferenceSession(self.model_path, providers=providers)
        self.input_name   = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        print(f'ONNX inputs : {self.input_name}')
        print(f'ONNX outputs: {self.output_names}')
        print(f'Active provider: {self.session.get_providers()[0]}')
        self.model = self.session

    def perform_inference(self, image: np.ndarray):
        h, w  = image.shape[:2]
        tile  = Image.fromarray(image).resize((self.image_size, self.image_size), Image.BILINEAR)
        tile  = np.array(tile, dtype=np.float32) / 255.0
        mean  = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std   = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        tile  = ((tile - mean) / std).transpose(2, 0, 1)[None]
        self._original_predictions = self.session.run(self.output_names, {self.input_name: tile})
        self._tile_hw = (h, w)

    def convert_original_predictions(self, shift_amount=None, full_shape=None):
      if shift_amount is None:
          shift_amount = [0, 0]

      dets   = self._original_predictions[0][0]   # [300, 4]  cx,cy,w,h normalised
      labels = self._original_predictions[1][0]   # [300, 1]  raw logits
      tile_h, tile_w = self._tile_hw

      # Confidence = sigmoid(logit)
      scores = 1.0 / (1.0 + np.exp(-labels[:, 0]))   # [300]

      object_prediction_list = []
      for box, score in zip(dets, scores):
          if score < self.confidence_threshold:
              continue
          cx, cy, bw, bh = box
          x1 = (cx - bw / 2) * tile_w
          y1 = (cy - bh / 2) * tile_h
          x2 = (cx + bw / 2) * tile_w
          y2 = (cy + bh / 2) * tile_h

          # Clip to tile boundaries
          x1, y1 = max(0.0, x1), max(0.0, y1)
          x2, y2 = min(float(tile_w), x2), min(float(tile_h), y2)

          if x2 <= x1 or y2 <= y1:
              continue

          # Apply tile offset to convert tile-local → full image coordinates
          x1 += shift_amount[0]
          y1 += shift_amount[1]
          x2 += shift_amount[0]
          y2 += shift_amount[1]

          category_name = self.category_mapping.get("0", "object")
          object_prediction_list.append(
              ObjectPrediction(
                  bbox=[x1, y1, x2, y2],
                  score=float(score),
                  category_id=0,
                  category_name=category_name,
                  shift_amount=[0, 0],   # already applied above
                  full_shape=full_shape,
              )
          )
      self._object_prediction_list_per_image = [object_prediction_list]


def sahi_result_to_sv_detections(result) -> sv.Detections:
    """Convert SAHI PredictionResult → supervision.Detections."""
    preds = result.object_prediction_list
    if not preds:
        return sv.Detections.empty()
    xyxy       = np.array([[p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy]
                            for p in preds], dtype=np.float32)
    confidence = np.array([p.score.value for p in preds], dtype=np.float32)
    class_ids  = np.array([p.category.id  for p in preds], dtype=int)
    return sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_ids)


DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Loading ONNX model on {DEVICE} at base threshold {SWEEP_BASE_THRESHOLD} ...')
detection_model = RFDetrOnnxDetectionModel(
    model_path=ONNX_MODEL_PATH,
    confidence_threshold=SWEEP_BASE_THRESHOLD,   # low — captures all candidates
    category_mapping=CATEGORY_MAPPING,
    image_size=IMAGE_SIZE,
    device=DEVICE,
)
detection_model.load_model()
print('Model loaded.\n')

In [ ]:
# load the Dataset
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=TEST_IMAGES_DIR,
    annotations_path=ANNOTATIONS_PATH,
)
print(f'Dataset loaded: {len(ds)} images, classes: {ds.classes}')

In [ ]:
## Run inference ONCE at SWEEP_BASE_THRESHOLD. Cell 12
## Raw detections are saved; the sweep below re-filters them without re-running inference.
from tqdm import tqdm
from PIL import Image

targets       = []
all_raw_preds = []   # detections at base threshold, one sv.Detections per image

for path, image, annotations in tqdm(ds, desc=f'Inference (base threshold={SWEEP_BASE_THRESHOLD})'):
    image_pil  = Image.open(path)
    result     = get_sliced_prediction(
        image_pil, detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )
    detections = sahi_result_to_sv_detections(result)
    if detections.class_id is not None:
        detections.class_id = np.clip(detections.class_id, 0, len(CLASS_NAMES) - 1)
    targets.append(annotations)
    all_raw_preds.append(detections)

print(f'\nInference complete. Raw detections per image (mean): '
      f'{sum(len(d) for d in all_raw_preds) / len(all_raw_preds):.1f}')

In [ ]:
## Sweep over SWEEP_THRESHOLDS.
## For each threshold: filter raw detections, compute TP/FP/FN,
## plot and save a confusion matrix, then write a summary CSV.
## mAP50 is computed ONCE on all_raw_preds before the loop.

import numpy as np
import matplotlib.pyplot as plt
import torch, torchvision, os, shutil, csv
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix
from supervision.metrics import MeanAveragePrecision


def match_detections(ground_truth, predictions, iou_threshold=0.5):
    if len(ground_truth) == 0 and len(predictions) == 0:
        return np.array([], int), np.array([], int), np.array([], int)
    elif len(ground_truth) == 0:
        return np.array([], int), np.array([], int), np.arange(len(predictions), dtype=int)
    elif len(predictions) == 0:
        return np.array([], int), np.array([], int), np.array([], int)

    iou_matrix = torchvision.ops.box_iou(
        torch.from_numpy(ground_truth.xyxy),
        torch.from_numpy(predictions.xyxy)
    ).numpy()
    gt_idx_arr, pred_idx_arr = linear_sum_assignment(-iou_matrix)

    matched_gt, matched_pred, matched_pred_set = [], [], set()
    for g, p in zip(gt_idx_arr, pred_idx_arr):
        if iou_matrix[g, p] >= iou_threshold:
            matched_gt.append(g)
            matched_pred.append(p)
            matched_pred_set.add(p)

    fp = [j for j in range(len(predictions)) if j not in matched_pred_set]
    return np.array(matched_gt, int), np.array(matched_pred, int), np.array(fp, int)


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Compute mAP ONCE on all raw predictions before any threshold filtering
map_result = MeanAveragePrecision().update(all_raw_preds, targets).compute()
MAP50 = map_result.map50
print(f'mAP@50:    {MAP50:.4f}')
print(f'mAP@50:95: {map_result.map50_95:.4f}')

csv_rows = []

for thresh in SWEEP_THRESHOLDS:
    print(f'\n{"="*62}')
    print(f'  Confidence threshold = {thresh:.2f}')
    print(f'{"="*62}')

    # Filter raw detections at this threshold
    filtered_preds = []
    for det in all_raw_preds:
        if det.confidence is None or len(det) == 0:
            filtered_preds.append(sv.Detections.empty())
        else:
            mask = det.confidence >= thresh
            filtered_preds.append(det[mask])

    n_total = sum(len(d) for d in filtered_preds)
    print(f'  Detections surviving filter: {n_total}  '
          f'(mean {n_total/max(len(filtered_preds),1):.1f}/image)')

    # TP / FP / FN
    total_tp = total_fp = total_fn = 0
    true_labels = []
    predicted_labels = []

    for gt_det, pred_det in zip(targets, filtered_preds):
        if gt_det is None or pred_det is None:
            continue
        matched_gt, matched_pred, fp_idx = match_detections(
            gt_det, pred_det, IOU_THRESHOLD
        )
        tp = len(matched_gt)
        fp = len(fp_idx)
        fn = len(gt_det) - tp
        total_tp += tp
        total_fp += fp
        total_fn += fn

        for g, p in zip(matched_gt, matched_pred):
            true_labels.append(gt_det.class_id[g])
            predicted_labels.append(pred_det.class_id[p])
        for p in fp_idx:
            true_labels.append(-1)
            predicted_labels.append(pred_det.class_id[p])
        for g in list(set(range(len(gt_det))) - set(matched_gt.tolist())):
            true_labels.append(gt_det.class_id[g])
            predicted_labels.append(-1)

    precision   = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall      = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1          = (2 * precision * recall / (precision + recall)
                   if (precision + recall) > 0 else 0.0)
    fp_tp_ratio = total_fp / total_tp if total_tp > 0 else float('inf')

    print(f'  TP={total_tp}  FP={total_fp}  FN={total_fn}')
    print(f'  Precision={precision:.4f}  Recall={recall:.4f}  '
          f'F1={f1:.4f}  FP/TP={fp_tp_ratio:.4f}')

    csv_rows.append({
        'Threshold':   thresh,
        'mAP50':       round(MAP50, 6),
        'FP':          total_fp,
        'TP':          total_tp,
        'FN':          total_fn,
        'Precision':   round(precision, 6),
        'Recall':      round(recall, 6),
        'F1':          round(f1, 6),
        'FP_TP_Ratio': round(fp_tp_ratio, 6) if fp_tp_ratio != float('inf') else 'inf',
    })

    # Confusion matrix
    all_cls       = sorted([l for l in set(true_labels + predicted_labels) if l != -1])
    unique_labels = all_cls + ([-1] if -1 in set(true_labels + predicted_labels) else [])
    label_to_idx  = {label: i for i, label in enumerate(unique_labels)}
    true_idx      = [label_to_idx[l] for l in true_labels]
    pred_idx_list = [label_to_idx[l] for l in predicted_labels]
    cm            = confusion_matrix(true_idx, pred_idx_list,
                                     labels=list(label_to_idx.values()))

    cm_labels = [None] * len(unique_labels)
    for label, idx in label_to_idx.items():
        if label == -1:
            cm_labels[idx] = 'No Object'
        else:
            try:
                cm_labels[idx] = ds.classes[label]
            except IndexError:
                cm_labels[idx] = f'Class {label}'

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]),
        xticklabels=cm_labels, yticklabels=cm_labels,
        title=(f'conf={thresh:.2f}  IoU={IOU_THRESHOLD}  '
               f'mAP@50={MAP50:.3f}  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}'),
        ylabel='True label', xlabel='Predicted label'
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    fmt     = '.2f' if cm.dtype.kind == 'f' else 'd'
    thresh_v = cm.max() / 2. if cm.max() > 0 else 1
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha='center', va='center',
                    color='white' if cm[i, j] > thresh_v else 'black')
    fig.tight_layout()

    conf_str   = f'{thresh:.2f}'.replace('.', '_')
    local_path = f'{OUTPUT_DIR}/confusion_matrix_conf_{conf_str}.png'
    drive_path = f'{DRIVE_OUTPUT}/confusion_matrix_conf_{conf_str}.png'
    plt.savefig(local_path)
    plt.show()
    try:
        shutil.copy2(local_path, drive_path)
    except Exception as e:
        print(f'  Could not copy confusion matrix to Drive: {e}')

# Write CSV
csv_columns = ['Threshold', 'mAP50', 'FP', 'TP', 'FN',
               'Precision', 'Recall', 'F1', 'FP_TP_Ratio']
with open(CSV_PATH, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(csv_rows)

drive_csv = f'{DRIVE_OUTPUT}/threshold_sweep_RF-DETR_large_detector_extended.csv'
try:
    shutil.copy2(CSV_PATH, drive_csv)
    print(f'\nCSV saved to Drive: {drive_csv}')
except Exception as e:
    print(f'\nCSV saved locally only ({CSV_PATH}): {e}')

# Print summary table
header = (f'  {"Thresh":>7}  {"mAP50":>7}  {"TP":>7}  {"FP":>7}  '
          f'{"FN":>7}  {"Prec":>7}  {"Rec":>7}  {"F1":>7}  {"FP/TP":>8}')
print(f'\n{"-" * len(header)}')
print(header)
print(f'{"-" * len(header)}')
for row in csv_rows:
    fpratio = f"{row['FP_TP_Ratio']:.4f}" if row['FP_TP_Ratio'] != 'inf' else '     inf'
    print(f"  {row['Threshold']:>7.2f}  {row['mAP50']:>7.4f}  {row['TP']:>7d}  "
          f"{row['FP']:>7d}  {row['FN']:>7d}  {row['Precision']:>7.4f}  "
          f"{row['Recall']:>7.4f}  {row['F1']:>7.4f}  {fpratio:>8}")
print(f'{"-" * len(header)}')

In [ ]:
## Single-image visualisation at a chosen threshold.
## Change VISUALISE_THRESHOLD and IMAGE_INDEX as needed.
VISUALISE_THRESHOLD = 0.45
IMAGE_INDEX         = 51

from PIL import Image

path, image, annotations = ds[IMAGE_INDEX]
image = Image.open(path)

result = get_sliced_prediction(
    image, detection_model,
    slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
    overlap_height_ratio=OVERLAP_RATIO, overlap_width_ratio=OVERLAP_RATIO,
    verbose=0,
)
detections = sahi_result_to_sv_detections(result)
if detections.confidence is not None and len(detections) > 0:
    detections = detections[detections.confidence >= VISUALISE_THRESHOLD]
if detections.class_id is not None:
    detections.class_id = np.clip(detections.class_id, 0, len(CLASS_NAMES) - 1)

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness  = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
color = sv.ColorPalette.from_hex([
    '#ffff00', '#ff9b00', '#ff66ff', '#3399ff', '#ff66b2', '#ff8080',
    '#b266ff', '#9999ff', '#66ffff', '#33ff99', '#66ff66', '#99ff00'
])
bbox_annotator  = sv.BoxAnnotator(color=color, thickness=thickness)
label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.BLACK,
                                     text_scale=text_scale)

annotations_labels = [f'{ds.classes[cid]}' for cid in annotations.class_id]
detections_labels  = [f'{ds.classes[cid]} {conf:.2f}'
                      for cid, conf in zip(detections.class_id, detections.confidence)]

ann_img = bbox_annotator.annotate(image.copy(), annotations)
ann_img = label_annotator.annotate(ann_img, annotations, annotations_labels)
det_img = bbox_annotator.annotate(image.copy(), detections)
det_img = label_annotator.annotate(det_img, detections, detections_labels)

sv.plot_images_grid(images=[ann_img, det_img], grid_size=(1, 2),
                   titles=['Ground Truth', f'ONNX + SAHI  conf>={VISUALISE_THRESHOLD}'])

In [ ]:
## Full dataset inference — annotated images + COCO predictions JSON.
## Uses VISUALISE_THRESHOLD from the cell above for the saved annotated images.
## Set SAVE_THRESHOLD here independently if you prefer.
SAVE_THRESHOLD = 0.45

import os, json, shutil
from PIL import Image
from tqdm import tqdm
from datetime import datetime

ANNOTATED_IMAGES_DIR  = os.path.join(OUTPUT_DIR, 'annotated_images')
COCO_ANNOTATIONS_PATH = os.path.join(OUTPUT_DIR, 'predictions_coco.json')
os.makedirs(ANNOTATED_IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
test_images = sorted([f for f in os.listdir(TEST_IMAGES_DIR)
                      if f.lower().endswith(image_extensions)])
print(f'Found {len(test_images)} images. Saving detections at conf >= {SAVE_THRESHOLD}')

coco_data = {
    'info': {'description': 'RF-DETR ONNX + SAHI', 'date_created': datetime.now().isoformat(),
             'model': 'RFDETRLarge ONNX', 'threshold': SAVE_THRESHOLD, 'slice_size': SLICE_SIZE},
    'images': [], 'annotations': [],
    'categories': [{'id': i, 'name': n, 'supercategory': 'none'}
                   for i, n in enumerate(CLASS_NAMES)],
}
annotation_id = image_id = 0

color = sv.ColorPalette.from_hex([
    '#ffff00', '#ff9b00', '#ff66ff', '#3399ff', '#ff66b2', '#ff8080',
    '#b266ff', '#9999ff', '#66ffff', '#33ff99', '#66ff66', '#99ff00'
])

for img_filename in tqdm(test_images, desc='Saving annotated images'):
    img_path  = os.path.join(TEST_IMAGES_DIR, img_filename)
    image_pil = Image.open(img_path).convert('RGB')
    width, height = image_pil.size

    # Re-use raw predictions collected in Cell 13, filtered at SAVE_THRESHOLD
    img_idx = test_images.index(img_filename)
    det     = all_raw_preds[img_idx]
    if det.confidence is not None and len(det) > 0:
        det = det[det.confidence >= SAVE_THRESHOLD]
    if det.class_id is not None:
        det.class_id = np.clip(det.class_id, 0, len(CLASS_NAMES) - 1)

    coco_data['images'].append({'id': image_id, 'file_name': img_filename,
                                 'width': width, 'height': height})
    if len(det) > 0:
        for i in range(len(det)):
            x1, y1, x2, y2 = det.xyxy[i]
            bw, bh = x2 - x1, y2 - y1
            coco_data['annotations'].append({
                'id': annotation_id, 'image_id': image_id,
                'category_id': int(det.class_id[i]) if det.class_id is not None else 0,
                'bbox': [float(x1), float(y1), float(bw), float(bh)],
                'area': float(bw * bh),
                'score': float(det.confidence[i]) if det.confidence is not None else 1.0,
                'iscrowd': 0,
            })
            annotation_id += 1

    text_scale = sv.calculate_optimal_text_scale(resolution_wh=(width, height))
    thickness  = sv.calculate_optimal_line_thickness(resolution_wh=(width, height))
    bbox_annotator  = sv.BoxAnnotator(color=color, thickness=thickness)
    label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.BLACK,
                                         text_scale=text_scale, smart_position=True)
    ann_img = image_pil.copy()
    if len(det) > 0:
        labels  = [f'object {c:.2f}' if c else 'object'
                   for c in (det.confidence if det.confidence is not None
                             else [None] * len(det))]
        ann_img = bbox_annotator.annotate(ann_img, det)
        ann_img = label_annotator.annotate(ann_img, det, labels)
    ann_img.save(os.path.join(ANNOTATED_IMAGES_DIR, img_filename))
    image_id += 1

with open(COCO_ANNOTATIONS_PATH, 'w') as f:
    json.dump(coco_data, f, indent=2)

print(f'\n✅ Done!')
print(f'   - Processed {len(test_images)} images at conf >= {SAVE_THRESHOLD}')
print(f'   - Found {annotation_id} total detections')
print(f'   - Annotated images: {ANNOTATED_IMAGES_DIR}')
print(f'   - COCO JSON:        {COCO_ANNOTATIONS_PATH}')

try:
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f'   - Copied to Drive: {DRIVE_OUTPUT}')
except Exception as e:
    print(f'   - Could not copy to Drive: {e}')

In [ ]:
# find images with the most detections
import json
from collections import Counter

with open(COCO_ANNOTATIONS_PATH) as f:
    coco_out = json.load(f)

total    = len(coco_out['annotations'])
n_images = len(coco_out['images'])
print(f'Total detections:          {total}')
print(f'Images:                    {n_images}')
print(f'Mean detections per image: {total/n_images:.1f}')

counts = Counter(a['image_id'] for a in coco_out['annotations'])
print('\nTop 5 images by detection count:')
for img_id, count in counts.most_common(5):
    fname = next(i['file_name'] for i in coco_out['images'] if i['id'] == img_id)
    print(f'  {fname}: {count} detections')

In [ ]:
# show one of the annotated images
from IPython.display import display
from PIL import Image

sample = os.path.join(ANNOTATED_IMAGES_DIR, test_images[0])
if os.path.exists(sample):
    display(Image.open(sample))
    print(f'\n📸 Preview: {test_images[0]}')

In [ ]:
# Check actual object sizes in your test set
import numpy as np
widths = []
heights = []
for path, img, ann in ds:
    for box in ann.xyxy:
        widths.append(box[2] - box[0])
        heights.append(box[3] - box[1])
print(f'Object width  — mean: {np.mean(widths):.1f}px  median: {np.median(widths):.1f}px  min: {np.min(widths):.1f}px')
print(f'Object height — mean: {np.mean(heights):.1f}px  median: {np.median(heights):.1f}px  min: {np.min(heights):.1f}px')